## בסיס שיתוף פעולה: branch ו-merge conflict אחד, מבוקר

**Branch** הוא קו התפתחות נפרד של הריפו — עותק-עבודה של ההיסטוריה שאפשר לנסות עליו דברים בלי לגעת ב-‎`main`‎. אם הניסוי הצליח, ממזגים (‎`merge`‎) אותו בחזרה; אם לא, מוחקים את ה-branch, ו-‎`main`‎ נשאר בדיוק כפי שהיה.

זה הופך שימושי במיוחד כששני אנשים (או שתי גרסאות של עצמכם) עורכים את **אותו קובץ, באותה שורה**, על שני branches שונים. כשמנסים למזג, Git לא יודע איזה שינוי "נכון" — הוא מסמן **merge conflict**, ומחכה שתחליטו בעצמכם.

קובץ עם קונפליקט מכיל סימונים מיוחדים:

```
<<<<<<< HEAD
התוכן ב-branch שאתם נמצאים עליו כרגע
=======
התוכן ב-branch שמנסים למזג פנימה
>>>>>>> partner
```

הפתרון: לערוך את הקובץ ידנית, לבחור (או לשלב) את התוכן הרצוי, למחוק את שלושת סימוני ה-‎`<<<`‎/‎`===`‎/‎`>>>`‎, ואז ‎`git add`‎ + ‎`git commit`‎ רגילים כדי לסיים את המיזוג.

### דוגמה: שני שותפי מעבדה, אותה שורה

שני שותפים עובדים על סקריפט קצר, ‎`cooling.py`‎, שמחשב את הטמפרטורה של גוף מתקרר אחרי ‎`n`‎ דקות לפי $T = 20 + (T_0 - 20)(1 - k)^n$. כל אחד מהם מדד את קבוע הקירור ‎`k`‎ בניסוי משלו. שותף א׳ (על ‎`main`‎) מעדכן את השורה של ‎`k`‎ לערך 0.10. שותף ב׳ (על branch נפרד, ‎`partner`‎, שהסתעף מ**אותה** נקודת התחלה) מעדכן **את אותה שורה בדיוק** לערך 0.12.

In [ ]:
import tempfile, os

workdir = tempfile.mkdtemp(prefix="git_branch_")
os.chdir(workdir)

!git init -q -b main
!git config user.email "student@example.com"
!git config user.name "Student"
!git config color.ui false

In [ ]:
%%writefile cooling.py
T0 = 90.0   # initial temperature (C)
k = 0.05    # cooling constant (1/min)
n = 10      # minutes
T = 20 + (T0 - 20) * (1 - k)**n
print(f"T after {n} min: {T:.1f} C")

In [ ]:
!git add cooling.py
!git commit -q -m "סקריפט קירור ראשוני"
!git branch partner

‎`git branch partner`‎ יצר branch חדש בשם ‎`partner`‎, מאותה נקודה בדיוק שבה אנחנו עכשיו. שם של branch, כמו ‎`main`‎ או ‎`partner`‎, הוא רק שם שמצביע על commit, כמו משתנה שמצביע על אובייקט (שבוע 5), וגם ‎`HEAD`‎ הוא שם כזה: הוא מצביע על ה-commit שעליו אתם עובדים כרגע. עכשיו, על ‎`main`‎, שותף א׳ מעדכן את ‎`k`‎:

In [ ]:
%%writefile cooling.py
T0 = 90.0   # initial temperature (C)
k = 0.10    # cooling constant (1/min)
n = 10      # minutes
T = 20 + (T0 - 20) * (1 - k)**n
print(f"T after {n} min: {T:.1f} C")

In [ ]:
!git add cooling.py
!git commit -q -m "עדכון k לפי המדידה של שותף א׳"

עכשיו עוברים ל-branch ‎`partner`‎, **מהנקודה המקורית** (בלי העדכון של שותף א׳), ומעדכנים שם את ‎`k`‎ לערך שמדד שותף ב׳, באותה שורה:

In [ ]:
!git checkout -q partner

In [ ]:
%%writefile cooling.py
T0 = 90.0   # initial temperature (C)
k = 0.12    # cooling constant (1/min)
n = 10      # minutes
T = 20 + (T0 - 20) * (1 - k)**n
print(f"T after {n} min: {T:.1f} C")

In [ ]:
!git add cooling.py
!git commit -q -m "עדכון k לפי המדידה של שותף ב׳"

עכשיו חוזרים ל-‎`main`‎ ומנסים למזג את ‎`partner`‎ פנימה:

In [ ]:
!git checkout -q main
!git merge partner -m "merge partner into main"

‎`CONFLICT`‎ — בדיוק כמו שציפינו: שני ה-branches שינו את אותה שורה בדרכים שונות, ו-Git לא יכול להחליט לבד. נסתכל על הקובץ:

In [ ]:
!git status --short
!cat cooling.py

‎`<<<<<<< HEAD`‎ עד ‎`=======`‎ הוא מה שהיה ב-‎`main`‎ (‎`k = 0.10`‎); מ-‎`=======`‎ עד ‎`>>>>>>> partner`‎ הוא מה שהיה ב-‎`partner`‎ (‎`k = 0.12`‎). שאר השורות זהות בשני ה-branches, ולכן Git מיזג אותן בלי קונפליקט. כאן אי אפשר לשלב את שתי הגרסאות: יש לבחור ערך אחד. נניח שהשותפים החליטו לשמור את 0.12 ולתעד בהערה את הערך שהיה ב-‎`main`‎. נערוך את הקובץ (ידנית, כמו שהייתם עושים בעורך קוד), ונסיים את המיזוג:

In [ ]:
%%writefile cooling.py
T0 = 90.0   # initial temperature (C)
k = 0.12    # cooling constant (1/min); main had 0.10
n = 10      # minutes
T = 20 + (T0 - 20) * (1 - k)**n
print(f"T after {n} min: {T:.1f} C")

In [ ]:
!git add cooling.py
!git commit -q --no-edit
!git log --oneline --graph --all

‎`git commit --no-edit`‎ (אחרי שפתרנו את הקונפליקט וסימנו את זה עם ‎`git add`‎) מסיים את המיזוג עם הודעת ברירת המחדל. גרף ההיסטוריה (‎`--graph --all`‎) מראה בבירור כיצד שני קווי ההתפתחות (‎`main`‎ ו-‎`partner`‎) התאחדו לקומיט מיזוג משותף אחד.

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "בסימוני קונפליקט, מה נמצא בין <<<<<<< HEAD לבין =======?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "התוכן מה-branch שממנו מגיעים לתוך המיזוג (partner, במקרה שלנו: k = 0.12)", "correct": False, "feedback": "לא — זה בדיוק ההפך. זה מה שנמצא אחרי =======."},
            {"answer": "התוכן מה-branch שאתם נמצאים עליו כרגע (main, במקרה שלנו: k = 0.10)", "correct": True, "feedback": "נכון — HEAD מציין את הנקודה שבה אתם נמצאים עכשיו."},
            {"answer": "שורות שנמחקו לגמרי ולא יופיעו בגרסה הסופית", "correct": False, "feedback": "לא — שתי הגרסאות עדיין קיימות, וצריך לבחור/לשלב ביניהן ידנית."},
            {"answer": "הערות שGit הוסיף אוטומטית ואינן חלק מהקוד עצמו", "correct": False, "feedback": "רק סימוני <<<</===/>>>> עצמם הם תוספת של Git; התוכן ביניהם הוא קוד אמיתי משני ה-branches, שצריך לערוך."}
        ]
    },
    {
        "question": "מדוע בדוגמה למעלה נוצר קונפליקט, ולא מיזוג אוטומטי חלק?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "כי שני ה-branches שינו קבצים שונים לגמרי", "correct": False, "feedback": "לא — בדיוק ההפך: אם היו משנים קבצים שונים, Git היה ממזג אוטומטית בלי בעיה."},
            {"answer": "כי שני ה-branches שינו את אותן שורות באותו קובץ, בדרכים שונות זו מזו", "correct": True, "feedback": "נכון — זה בדיוק מה שגורם ל-Git לא לדעת איזו גרסה לבחור."},
            {"answer": "כי לא הרצתם git add לפני git merge", "correct": False, "feedback": "לא קשור — git add דרוש רק אחרי פתרון הקונפליקט, לא לפני הניסיון למזג."},
            {"answer": "קונפליקטים קורים תמיד, בכל מיזוג, ללא קשר לתוכן השינויים", "correct": False, "feedback": "לא נכון — רוב המיזוגים חלקים לגמרי; קונפליקט קורה רק כששתי הגרסאות מתנגשות באותן שורות."}
        ]
    },
    {
        "question": "מה יוצרת הפקודה git branch partner?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "עותק נוסף של כל קבצי הפרויקט, בתיקייה חדשה", "correct": False, "feedback": "לא — שום קובץ אינו מועתק; נוצר רק שם חדש."},
            {"answer": "שם חדש שמצביע על ה-commit הנוכחי, כמו משתנה שמצביע על אובייקט", "correct": True, "feedback": "נכון — branch הוא שם שמצביע על commit, ו-HEAD הוא שם שמצביע על המקום שבו אתם עובדים."},
            {"answer": "commit חדש בהיסטוריה", "correct": False, "feedback": "לא — git branch אינו יוצר commit; ההיסטוריה לא משתנה."},
            {"answer": "ריפו מרוחק חדש ב-GitHub", "correct": False, "feedback": "לא — branch נוצר בריפו המקומי בלבד."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

בנו תרחיש קונפליקט דומה בעצמכם: קובץ ‎`plot_config.py`‎ עם השורה ‎`title = "Cooling curve"`‎. על ‎`main`‎, שנו את הכותרת ל-‎`"Cooling curve, k = 0.12"`‎. על branch נפרד בשם ‎`alt`‎, שהסתעף **מאותה נקודה**, שנו את אותה שורה לכותרת ‎`"Coffee cup cooling"`‎. מזגו את ‎`alt`‎ לתוך ‎`main`‎, פתרו את הקונפליקט (בחרו כותרת אחת, או שלבו), וסיימו את המיזוג.

In [ ]:
# workdir2 = tempfile.mkdtemp(prefix="git_branch_practice_")
# os.chdir(workdir2)
# !git init -q -b main
# !git config user.email "student@example.com"
# !git config user.name "Student"
# !git config color.ui false
#
# כתבו כאן את כל הרצף: קומיט בסיס, branch, שינוי על main, שינוי על alt, merge, פתרון

`````{admonition} פתרון
:class: dropdown, tip
```python
workdir2 = tempfile.mkdtemp(prefix="git_branch_practice_")
os.chdir(workdir2)
!git init -q -b main
!git config user.email "student@example.com"
!git config user.name "Student"
!git config color.ui false
```
```python
%%writefile plot_config.py
title = "Cooling curve"
```
```python
!git add plot_config.py
!git commit -q -m "כותרת ראשונית"
!git branch alt
```
```python
%%writefile plot_config.py
title = "Cooling curve, k = 0.12"
```
```python
!git add plot_config.py
!git commit -q -m "main: עדכון כותרת"
```
```python
!git checkout -q alt
```
```python
%%writefile plot_config.py
title = "Coffee cup cooling"
```
```python
!git add plot_config.py
!git commit -q -m "alt: עדכון כותרת"
```
```python
!git checkout -q main
!git merge alt -m "merge alt into main"
!cat plot_config.py   # shows the conflict markers
```
```python
%%writefile plot_config.py
title = "Coffee cup cooling, k = 0.12"
```
```python
!git add plot_config.py
!git commit -q --no-edit
!git log --oneline --graph --all
```
`````